# LangGraph 101

[LLMs](https://python.langchain.com/docs/concepts/chat_models/) make it possible to embed intelligence into a new class of applications. [LangGraph](https://langchain-ai.github.io/langgraph/) is a framework to help build applications with LLMs. Here, we will overview the basics of LangGraph, explain its benefits, show how to use it to build workflows / agents, and show how it works with [LangChain](https://www.langchain.com/) / [LangSmith](https://docs.smith.langchain.com/).

![ecosystem](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/ecosystem.png?raw=1)

## Chat models

[Chat models](https://python.langchain.com/docs/concepts/chat_models/) are the foundation of LLM applications. They are typically accessed through a chat interface that takes a list of [messages](https://python.langchain.com/docs/concepts/messages/) as input and returns a [message](https://python.langchain.com/docs/concepts/messages/) as output. LangChain provides [a standardized interface for chat models](https://python.langchain.com/api_reference/langchain/chat_models/langchain.chat_models.base.init_chat_model.html), making it easy to [access many different providers](https://python.langchain.com/docs/integrations/chat/).

In [10]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

ModuleNotFoundError: No module named 'dotenv'

In [5]:
# prompt: pip install langchain_google_genai

!pip install langchain-google-genai

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=userdata.get('GOOGLE_API_KEY'))

## Running the model

The `init_chat_model` interface provides [standardized](https://python.langchain.com/docs/concepts/runnables/) methods for using chat models, which include:
- `invoke()`: A single input is transformed into an output.
- `stream()`: Outputs are [streamed](https://python.langchain.com/docs/concepts/streaming/#stream-and-astream) as they are produced.

In [12]:
result = llm.invoke("What is an agent?")

In [13]:
type(result)

langchain_core.messages.ai.AIMessage

In [14]:
from rich.markdown import Markdown
Markdown(result.content)

The term "agent" has different meanings depending on the context. Here's a breakdown of common interpretations:    

1. In General Terms (Broad Definition):                                                                            

 • An agent is someone or something that acts or has the power to act. It's an entity that can perform actions and 
   potentially influence its environment. This is a very broad definition.                                         

2. In Business and Legal Contexts:                                                                                 

 • An agent is a person or company authorized to act on behalf of another (the principal).  The agent's actions    
   bind the principal, meaning the principal is responsible for the agent's actions within the scope of their      
   authority. Key aspects include:                                                                                 
    • Authority: The specific powers granted to the agent by the principal. This can be express (explicitly stated)
      or implied (reasonably inferred from the circumstances).                                                     
    • Fiduciary Duty:  The agent owes a duty of loyalty, good faith, and care to the principal.  They must act in  
      the principal's best interests.                                                                              
    • Examples:                                                                                                    
       • Real Estate Agent: Represents a buyer or seller of property.                                              
       • Insurance Agent: Sells insurance policies on behalf of an insurance company.                              
       • Literary Agent: Represents an author to publishers.                                                       
       • Talent Agent: Represents actors, musicians, athletes, etc.                                                
       • Travel Agent: Arranges travel plans for clients.                                                          

3. In Computer Science (AI & Software):                                                                            

 • An agent is an autonomous entity that perceives its environment through sensors and acts upon that environment  
   through actuators.  It's a key concept in artificial intelligence and software engineering.  Key characteristics
   include:                                                                                                        
    • Autonomy:  Operates without direct human intervention (at least to some degree).                             
    • Perception:  Gathers information about the environment.                                                      
    • Action:  Performs actions to achieve a goal.                                                                 
    • Goal-Oriented:  Designed to achieve specific objectives.                                                     
    • Examples:                                                                                                    
       • Software Agent: A program that performs tasks on behalf of a user (e.g., a web crawler, a chatbot).       
       • Intelligent Agent: A more sophisticated agent that uses AI techniques (e.g., machine learning) to adapt   
         and improve its performance.                                                                              
       • Robotic Agent: A physical robot that interacts with the real world.                                       

4. In Chemistry and Biology:                                                                                       

 • An agent is a substance or factor that causes a particular effect or change.                                    
    • Chemical Agent: A chemical substance that produces a specific reaction (e.g., a reducing agent, an oxidizing 
      agent).                                   

## Tools

[Tools](https://python.langchain.com/docs/concepts/tools/) are utilities that can be called by a chat model. In LangChain, creating tools can be done using the `@tool` decorator, which transforms Python functions into callable tools. It will automatically infer the tool's name, description, and expected arguments from the function definition. You can also use [Model Context Protocol (MCP) servers](https://github.com/langchain-ai/langchain-mcp-adapters) as LangChain-compatible tools.

In [15]:
from langchain.tools import tool

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in real app would send email
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

In [16]:
type(write_email)

langchain_core.tools.structured.StructuredTool

In [17]:
write_email.args

{'to': {'title': 'To', 'type': 'string'},
 'subject': {'title': 'Subject', 'type': 'string'},
 'content': {'title': 'Content', 'type': 'string'}}

In [18]:
Markdown(write_email.description)

Write and send an email.

## Tool Calling

Tools can be [called](https://python.langchain.com/docs/concepts/tool_calling/) by LLMs. When a tool is bound to the model, the model can choose to call the tool by returning a structured output with tool arguments. We use the `bind_tools` method to augment an LLM with tools.

![tool-img](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/tool_call_detail.png?raw=1)

Providers often have [parameters such as `tool_choice`](https://python.langchain.com/docs/how_to/tool_choice/) to enforce calling specific tools. `any` will select at least one of the tools.

In addition, we can [set `parallel_tool_calls=False`](https://python.langchain.com/docs/how_to/tool_calling_parallel/) to ensure the model will only call one tool at a time.

In [20]:
# Connect tools to a chat model
model_with_tools = llm.bind_tools([write_email], tool_choice="any")

# The model will now be able to call tools
output = model_with_tools.invoke("Draft a response to my boss (boss@company.ai) about tomorrow's meeting")

In [21]:
type(output)

langchain_core.messages.ai.AIMessage

In [22]:
output

AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_email', 'arguments': '{"to": "boss@company.ai", "content": "Hi Boss,\\n\\nI\'m looking forward to our meeting tomorrow.\\n\\nBest,\\n[Your Name]", "subject": "Tomorrow\'s Meeting"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--70941af0-f097-442f-a048-ca8a0fd248f2-0', tool_calls=[{'name': 'write_email', 'args': {'to': 'boss@company.ai', 'content': "Hi Boss,\n\nI'm looking forward to our meeting tomorrow.\n\nBest,\n[Your Name]", 'subject': "Tomorrow's Meeting"}, 'id': '98997078-3d23-433d-8e16-a8cd51a4273c', 'type': 'tool_call'}], usage_metadata={'input_tokens': 37, 'output_tokens': 37, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [23]:
# Extract tool calls and execute them
args = output.tool_calls[0]['args']
args

{'to': 'boss@company.ai',
 'content': "Hi Boss,\n\nI'm looking forward to our meeting tomorrow.\n\nBest,\n[Your Name]",
 'subject': "Tomorrow's Meeting"}

In [24]:
# Call the tool
result = write_email.invoke(args)
Markdown(result)

Email sent to boss@company.ai with subject 'Tomorrow's Meeting' and content: Hi Boss,                              

I'm looking forward to our meeting tomorrow.                                                                       

Best, [Your Name]

![basic_prompt](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/tool_call.png?raw=1)

## Workflows

There are many patterns for building applications with LLMs.

[We can embed LLM calls into pre-defined workflows](https://langchain-ai.github.io/langgraph/tutorials/workflows/), giving the system more agency to make decisions.

As an example, we could add a router step to determine whether to write an email or not.

![workflow_example](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/workflow_example.png?raw=1)

## Agents

We can further increase agency, allowing the LLM to dynamically direct its own tool usage.

[Agents](https://langchain-ai.github.io/langgraph/tutorials/workflows/) are typically implemented as tool calling in a loop, where the output of each tool call is used to inform the next action.

![agent_example](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/agent_example.png?raw=1)

Agents are well suited to open-ended problems where it's difficult to predict the *exact* steps needed in advance.

Workflows are often appropriate when the control flow can easily be defined in advance.

![workflow_v_agent](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/workflow_v_agent.png?raw=1)

## What is LangGraph?

[LangGraph](https://langchain-ai.github.io/langgraph/concepts/high_level/) provides low-level supporting infrastructure that sits underneath *any* workflow or agent.

It does not abstract prompts or architecture, and provides a few benefits:

- **Control**: Make it easy to define and / or combine agents and workflows.
- **Persistence**: Provide a way to persist the state of a graph, which enables both memory and human-in-the-loop.
- **Testing, Debugging, and Deployment**: Provide an easy onramp for testing, debugging, and deploying applications.

### Control

LangGraph lets you define your application as a graph with:

1. *State*: What information do we need to track over the course of the application?
2. *Nodes*: How do we want to update this information over the course of the application?
3. *Edges*: How do we want to connect these nodes together?

We can use the [`StateGraph` class](https://langchain-ai.github.io/langgraph/concepts/low_level/#graphs) to initialize a LangGraph graph with a [`State` object](https://langchain-ai.github.io/langgraph/concepts/low_level/#state).

`State` defines the schema for information we want to track over the course of the application.

This can be any object with `getattr()` in python, such as a dictionary, dataclass, or Pydantic object:

- TypeDict is fastest but doesn’t support defaults
- Dataclass is basically as fast, supports dot syntax `state.foo`, and has defaults.
- Pydantic is slower (especially with custom validators) but gives type validation.

In [25]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class StateSchema(TypedDict):
    request: str
    email: str

workflow = StateGraph(StateSchema)

ModuleNotFoundError: No module named 'langgraph'

Each node is simply a python function or typescript code. This gives us full control over the logic inside each node.

They receive the current state, and return a dictionary to update the state.

By default, [state keys are overwritten](https://langchain-ai.github.io/langgraph/how-tos/state-reducers/).

However, you can [define custom update logic](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers).

![nodes_edges](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/nodes_edges.png?raw=1)

In [ ]:
def write_email_node(state: StateSchema) -> StateSchema:
    # Imperative code that processes the request
    output = model_with_tools.invoke(state["request"])
    args = output.tool_calls[0]['args']
    email = write_email.invoke(args)
    return {"email": email}

Edges connect nodes together.

We specify the control flow by adding edges and nodes to our state graph.

In [ ]:
workflow = StateGraph(StateSchema)
workflow.add_node("write_email_node", write_email_node)
workflow.add_edge(START, "write_email_node")
workflow.add_edge("write_email_node", END)

app = workflow.compile()

In [ ]:
app.invoke({"request": "Draft a response to my boss (boss@company.ai) about tomorrow's meeting"})

Routing between nodes can be done [conditionally](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges) using a simple function.

The return value of this function is used as the name of the node (or list of nodes) to send the state to next.

You can optionally provide a dictionary that maps the `should_continue` output to the name of the next node.

In [ ]:
from typing import Literal
from langgraph.graph import MessagesState
from email_assistant.utils import show_graph

def call_llm(state: MessagesState) -> MessagesState:
    """Run LLM"""

    output = model_with_tools.invoke(state["messages"])
    return {"messages": [output]}

def run_tool(state: MessagesState):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        observation = write_email.invoke(tool_call["args"])
        result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
    return {"messages": result}

def should_continue(state: MessagesState) -> Literal["run_tool", "__end__"]:
    """Route to tool handler, or end if Done tool called"""

    # Get the last message
    messages = state["messages"]
    last_message = messages[-1]

    # If the last message is a tool call, check if it's a Done tool call
    if last_message.tool_calls:
        return "run_tool"
    # Otherwise, we stop (reply to the user)
    return END

workflow = StateGraph(MessagesState)
workflow.add_node("call_llm", call_llm)
workflow.add_node("run_tool", run_tool)
workflow.add_edge(START, "call_llm")
workflow.add_conditional_edges("call_llm", should_continue, {"run_tool": "run_tool", END: END})
workflow.add_edge("run_tool", END)

# Run the workflow
app = workflow.compile()

In [ ]:
show_graph(app)

In [ ]:
result = app.invoke({"messages": [{"role": "user", "content": "Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!"}]})
for m in result["messages"]:
    m.pretty_print()

With these low level components, you can build many many different workflows and agents. See [this tutorial](https://langchain-ai.github.io/langgraph/tutorials/workflows/)!

Because agents are such a common pattern, [LangGraph](https://langchain-ai.github.io/langgraph/tutorials/workflows/#pre-built) has [a pre-built agent](https://langchain-ai.github.io/langgraph/agents/overview/?ref=blog.langchain.dev#what-is-an-agent) abstraction.

With LangGraph's [pre-built method](https://langchain-ai.github.io/langgraph/tutorials/workflows/#pre-built), we just pass in the LLM, tools, and prompt.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[write_email],
    prompt="Respond to the user's request using the tools provided."
)

# Run the agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!"}]}
)

for m in result["messages"]:
    m.pretty_print()

### Persistence

#### Threads

It can be very useful to allow agents to pause during long running tasks.

LangGraph has a built-in persistence layer, implemented through checkpointers, to enable this.

When you compile graph with a checkpointer, the checkpointer saves a [checkpoint](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpoints) of the graph state at every step.

Checkpoints are saved to a thread, which can be accessed after graph execution completes.

![checkpointer](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/checkpoints.png?raw=1)

We compile the graph with a [checkpointer](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpointer-libraries).


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_react_agent(
    model=llm,
    tools=[write_email],
    prompt="Respond to the user's request using the tools provided.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}
result = agent.invoke({"messages": [{"role": "user", "content": "What are some good practices for writing emails?"}]}, config)


In [ ]:
# Get the latest state snapshot
config = {"configurable": {"thread_id": "1"}}
state = agent.get_state(config)
for message in state.values['messages']:
    message.pretty_print()

In [ ]:
# Continue the conversation
result = agent.invoke({"messages": [{"role": "user", "content": "Good, let's use lesson 3 to craft a response to my boss confirming that I want to attend Interrupt"}]}, config)
for m in result['messages']:
    m.pretty_print()

In [ ]:
# Continue the conversation
result = agent.invoke({"messages": [{"role": "user", "content": "I like this, let's write the email to boss@company.ai"}]}, config)
for m in result['messages']:
    m.pretty_print()

#### Interrupts

In LangGraph, we can also use [interrupts](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/wait-user-input/) to stop graph execution at specific points.

Often this is used to collect input from a user and continue execution with collected input.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    input: str
    user_feedback: str

def step_1(state):
    print("---Step 1---")
    pass

def human_feedback(state):
    print("---human_feedback---")
    feedback = interrupt("Please provide feedback:")
    return {"user_feedback": feedback}

def step_3(state):
    print("---Step 3---")
    pass

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("human_feedback", human_feedback)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "human_feedback")
builder.add_edge("human_feedback", "step_3")
builder.add_edge("step_3", END)

# Set up memory
memory = InMemorySaver()

# Add
graph = builder.compile(checkpointer=memory)

In [ ]:
show_graph(graph)

In [ ]:
# Input
initial_input = {"input": "hello world"}

# Thread
thread = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="updates"):
    print(event)
    print("\n")

To resume from an interrupt, we can use [the `Command` object](https://langchain-ai.github.io/langgraph/how-tos/command/).

We'll use it to resume the graph from the interrupted state, passing the value to return from the interrupt call to `resume`.

In [ ]:
# Continue the graph execution
for event in graph.stream(
    Command(resume="go to step 3!"),
    thread,
    stream_mode="updates",
):
    print(event)
    print("\n")

### Tracing

When we are using LangChain or LangGraph, LangSmith logging [will work out of the box](https://docs.smith.langchain.com/observability/how_to_guides/trace_with_langgraph) with the following environment variables set:

```
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY="<your-langsmith-api-key>"
```

Here is the LangSmith trace from above agent execution:

https://smith.langchain.com/public/6f77014f-d054-44ed-aa2c-8b06ceab689f/r

We can see that the agent is able to continue the conversation from the previous state because we used a checkpointer.

### Deployment

We can also deploy our graph using [LangGraph Platform](https://langchain-ai.github.io/langgraph/concepts/langgraph_platform/).

This creates a server [with an API](https://langchain-ai.github.io/langgraph/cloud/reference/api/api_ref.html) that we can use to interact with our graph and an interactive IDE, LangGraph [Studio](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/).

We simply need to ensure our project has [a structure](https://langchain-ai.github.io/langgraph/concepts/application_structure/) like this:

```
my-app/
├── src/email_assistant # all project code lies within here
│   └── langgraph101.py # code for constructing your graph
├── .env # environment variables
├── langgraph.json  # configuration file for LangGraph
└── pyproject.toml # dependencies for your project
```

The `langgraph.json` file specifies the dependencies, graphs, environment variables, and other settings required to start a LangGraph server.

To test this, let's deploy `langgraph_101.py`. We have it in our `langgraph.json` file in this repo:

```
 "langgraph101": "./src/email_assistant/langgraph_101.py:app",
```

For LangGraph Platform, there are a range of [deployment options](https://langchain-ai.github.io/langgraph/tutorials/deployment/):

* Local deployments can be started with `langgraph dev` from the root directory of the repo. Checkpoints are saved to the local filesystem.
* There are also various [self-hosted options](https://langchain-ai.github.io/langgraph/tutorials/deployment/#other-deployment-options).
* For hosted deployments, checkpoints are saved to Postgres using a postgres [checkpointer](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpointer-libraries).

Test:
```
Draft a response to my boss (boss@company.ai) confirming that I want to attent Interrupt!
```

Here we can see a visualization of the graph as well as the graph state in Studio.

![langgraph_studio](https://github.com/Anish-Koirala7/agents-from-scratch/blob/main/notebooks/img/langgraph_studio.png?raw=1)

Also, you can see API docs for the local deployment here:

http://127.0.0.1:2024/docs

In [26]:
%pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
